# CENG463 PA2

In this programming assignment, you will be dealing with word embeddings and neural networks. You will use Python for this task. You can use libraries such as `pandas`, `nltk`, `numpy` etc. for your implementations, or implement your own functions. However, you are expected to analyse and reason about your implementation and results. The assignment consists of 3 questions.

### IMPORTANT NOTE

Do not move or delete the given cells, only add cells inbetween the questions for your answers.

In [1]:
# UPDATE THIS CELL TO INSTALL NEEDED LIBRARIES.
# MAKE SURE TO ADD EVERYTHING THAT NEEDS TO BE INSTALLED IN THIS CELL!

# we will use pip to install packages - you can add others below
!pip install pandas
!pip install numpy
!pip install nltk
!pip install gensim
!pip install scikit-learn

# and import them here - you can add others below
import pandas as pd
import numpy as np
import nltk
import gensim

## Q1 - Word embeddings (50 points)

In this question, you will first train a Word2Vec model, then use it to represent and reason about user reviews.

### Q1.A - training (10 points)
Load the `user_review_train.csv` file shared with you. Using `Word2Vec` module of `gensim.models`, train a **skip-gram** Word2Vec model on the train data.

#### Notes and tips

- Use the given preprocessing function `preprocess_review`.

In [2]:
# PREPROCESSING FUNCTIONS GIVEN FOR YOU

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet')  
nltk.download('stopwords')
nltk.download('punkt_tab')

def preprocess_review(review):
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    sentences = sent_tokenize(review)
    
    lemmatized_review = []
    
    for sentence in sentences:
        tokenized_sentence = word_tokenize(sentence)
        lowercased_sentence = [token.lower() for token in tokenized_sentence]
        stopwords_removed_sentence = [token for token in lowercased_sentence if token not in stop_words]
        lemmatized_sentence = [lemmatizer.lemmatize(token) for token in stopwords_removed_sentence]
        
        lemmatized_review = lemmatized_review + lemmatized_sentence
    
    return lemmatized_review

[nltk_data] Downloading package wordnet to /Users/ovak/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/ovak/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/ovak/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [3]:
# Q1.A - implementation
# you can add cells below if needed
review_test = pd.read_csv('data/user_review_train.csv')
review_test["preprocessed_review"] = review_test["review"].apply(preprocess_review)
print(review_test["preprocessed_review"].head())

model = gensim.models.Word2Vec(sentences=review_test["preprocessed_review"], 
                               vector_size=100, 
                               window=5, 
                               min_count=5,
                               workers=4, 
                               sg=1,
                               epochs=10)
print("Word2Vec model trained successfully.")
print(f"Vocabulary size: {len(model.wv)}")

0    [worst, mobile, bought, ever, ,, battery, drai...
1    [worst, phone, everthey, changed, last, phone,...
2    ['m, telling, n't, buyi, 'm, totally, disappoi...
3                               [battery, level, worn]
4    ['s, hitting, problem, ..., phone, hanging, pr...
Name: preprocessed_review, dtype: object


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Word2Vec model trained successfully.
Vocabulary size: 2539


### Q1.B - word similarity (10 points)

Using the trained model, report the following:

- Similarity between "good" and "bad"
- Similar words to "good"
- Similar words to "bad"
- Similar words to "good" but not similar to "bad"
- Similar words to "good" but not similar to "bad"

and discuss the reported words and scores. Is it possible to identify specific good/bad features of the product that is being reviewed? What other words can be looked up to get more insight?

#### Notes and tips

- Check the [documentation](https://tedboy.github.io/nlps/generated/generated/gensim.models.Word2Vec.html) of `gensim.models.Word2Vec` to find relevant methods.

In [4]:
# Q1.B - implementation
# you can add cells below if needed

sim = model.wv.similarity('good', 'bad')  # Example similarity check
print("Similarity between 'good' and 'bad':", sim)

most_similar_words = model.wv.most_similar("good")
print("Words most similar to 'good':", most_similar_words)

most_similar_words = model.wv.most_similar("bad")
print("Words most similar to 'bad':", most_similar_words)


simGoodbutNotBad = model.wv.most_similar(positive=['good'], negative=['bad'], topn=5)
print("Words similar to 'good' but not 'bad':", simGoodbutNotBad)

simGoodbutNotBad = model.wv.most_similar(positive=['bad'], negative=['good'], topn=5)
print("Words similar to 'bad' but not 'good':", simGoodbutNotBad)

Similarity between 'good' and 'bad': 0.5292213
Words most similar to 'good': [('nice', 0.8296639323234558), ('satisfying', 0.8165647983551025), ('good.but', 0.7924157381057739), ('impressive', 0.78011554479599), ('good.overall', 0.7757611870765686), ('satisfactory', 0.7734183669090271), ('gr8', 0.7716116905212402), ('beautiful', 0.7698405385017395), ('good.no', 0.7629500031471252), ('handy', 0.7610340118408203)]
Words most similar to 'bad': [('pathetic', 0.6652039289474487), ('👎', 0.6314492225646973), ('poor', 0.6173127889633179), ('nd', 0.6092343330383301), ('amozon', 0.6058595180511475), ('bed', 0.600261926651001), ('worst', 0.5919132232666016), ('grt', 0.5891046524047852), ('nyc', 0.5867640376091003), ('disgusting', 0.5697413086891174)]
Words similar to 'good' but not 'bad': [('impressive', 0.4459158480167389), ('operation', 0.44486376643180847), ('costly', 0.43649065494537354), ('slim', 0.4289926290512085), ('excellent', 0.41865161061286926)]
Words similar to 'bad' but not 'good': 

### Q1.B - discussion
Write your discussion here

### Q1.C - representation (15 points)

An important use of word embeddings is representing "documents" (reviews in our case). For this question, before creating the representations, do the following:

- Randomly sample 2 reviews from sentiment label 0, refer to them as sent0_a and sent0_b.
- Randomly sample 2 reviews from sentiment label 1, refer to them as sent1_a and sent1_b.

After the sampling, follow these steps to represent each review:

- Preprocess the review with the given `preprocess_review` function.
- For each token in the review, fetch the vector of that token.
- Take the average of the token vectors in the review to represent that review.

Then, calculate and report the cosine similarity of the two vectors representing:
    - sent0_a and sent0_b
    - sent0_a and sent1_a
    - sent1_a and sent1_b

Does this representation work to capture the labels of the reviews? Do you think there is a better way to represent each review instead of taking the average of the word vectors? Discuss your findings with respect to these questions. Repeating the sampling process several times might give you a better insight.

#### Notes and tips

- You can use `numpy` for your calculations.

In [5]:
# Q1.C - implementation
# you can add cells below if needed
def get_rndm_review(sentiment):
    while True:
        index = np.random.randint(0, len(review_test))
        select_review = review_test["review"][index]
        if(review_test["sentiment"][index] == sentiment):
            print("Selected review index:", index)
            break
    return select_review


np.random.seed(42)
sent1_a = get_rndm_review(1)
sent1_b = get_rndm_review(1)
sent0_a = get_rndm_review(0)
sent0_b = get_rndm_review(0)

print("Positive Review 1:", sent1_a)
print("Positive Review 2:", sent1_b)      
print("Negative Review 1:", sent0_a)
print("Negative Review 2:", sent0_b)

Selected review index: 13418
Selected review index: 11964
Selected review index: 5734
Selected review index: 6265
Positive Review 1: A good quality phone. Everything as expected. Camera is nice and battery lasts the entire day if not used for Hotspot.
Positive Review 2: Nice Product
Negative Review 1: Waist of money every time heating problem no battery back and in a one year 3 time repair on customer service center so please don't buy this
Negative Review 2: Worst of money to buy this product


In [6]:
#preprocess reviews

preprocessed_positive_review_1 = preprocess_review(sent1_a)
preprocessed_positive_review_2 = preprocess_review(sent1_b)
preprocessed_negative_review_1 = preprocess_review(sent0_a)
preprocessed_negative_review_2 = preprocess_review(sent0_b)

print("Preprocessed Positive Review 1:", preprocessed_positive_review_1)
print("Preprocessed Positive Review 2:", preprocessed_positive_review_2)
print("Preprocessed Negative Review 1:", preprocessed_negative_review_1)
print("Preprocessed Negative Review 2:", preprocessed_negative_review_2)


Preprocessed Positive Review 1: ['good', 'quality', 'phone', '.', 'everything', 'expected', '.', 'camera', 'nice', 'battery', 'last', 'entire', 'day', 'used', 'hotspot', '.']
Preprocessed Positive Review 2: ['nice', 'product']
Preprocessed Negative Review 1: ['waist', 'money', 'every', 'time', 'heating', 'problem', 'battery', 'back', 'one', 'year', '3', 'time', 'repair', 'customer', 'service', 'center', 'please', "n't", 'buy']
Preprocessed Negative Review 2: ['worst', 'money', 'buy', 'product']


In [7]:
#For each token in the review, fetch the vector of that token.

def get_review_vector(review):
    review_vector = np.zeros((100,))
    valid_word_count = 0
    for token in review:
        if token in model.wv:  # Check if token exists
            review_vector += model.wv[token]
            valid_word_count += 1
    
    if valid_word_count > 0:  # Avoid division by zero
        review_vector /= valid_word_count
    return review_vector

positive_review_vector_1 = get_review_vector(preprocessed_positive_review_1)
positive_review_vector_2 = get_review_vector(preprocessed_positive_review_2)
negative_review_vector_1 = get_review_vector(preprocessed_negative_review_1)
negative_review_vector_2 = get_review_vector(preprocessed_negative_review_2)

print("Positive Review Vector 1:", positive_review_vector_1)
print("Positive Review Vector 2:", positive_review_vector_2)
print("Negative Review Vector 1:", negative_review_vector_1)
print("Negative Review Vector 2:", negative_review_vector_2)

Positive Review Vector 1: [-0.16806336  0.37637131 -0.10861687  0.04709469  0.14595189 -0.14320109
  0.18011969  0.43073764 -0.25063133 -0.31383413 -0.04297271 -0.22739388
 -0.05392634 -0.06230058  0.13298774 -0.20013528  0.05098843 -0.24621528
 -0.13720332 -0.48022329  0.08514286  0.07752537  0.05921794 -0.16440427
 -0.23493331 -0.04182373 -0.1803304  -0.2841548   0.0817248   0.07041412
  0.15925164  0.0054581   0.05875419 -0.25358499 -0.03031085  0.3247757
 -0.05622713 -0.07184022 -0.01919044 -0.28457731 -0.02292544 -0.11950382
 -0.11695808  0.1288065   0.09772115 -0.11393203 -0.06910875 -0.11083837
 -0.04153917  0.16609367  0.12369055 -0.04888701 -0.03405543  0.04749209
 -0.03802104 -0.08548408  0.10093711 -0.07658596 -0.2439767   0.0496143
  0.1667094  -0.04622719  0.18065852  0.08298544 -0.08696441  0.2537404
  0.23158637  0.15478625 -0.2163146   0.02619996  0.02570354 -0.00719001
  0.003169    0.04163181  0.19380504  0.14695549 -0.06485318  0.14479476
 -0.07729421 -0.14027718  0.

In [8]:
# Cosine similarty function

def cosine_similarity(vec1, vec2):
    cos_sim = np.dot(vec1, vec2)/(np.linalg.norm(vec1)*np.linalg.norm(vec2))
    return cos_sim

# Calculate cosine similarities
sim_pos_pos = cosine_similarity(positive_review_vector_1, positive_review_vector_2)
sim_pos_neg_1 = cosine_similarity(positive_review_vector_1, negative_review_vector_1)
sim_neg_neg = cosine_similarity(negative_review_vector_1, negative_review_vector_2)


print("Cosine Similarity between two positive reviews:", sim_pos_pos)
print("Cosine Similarity between positive and negative review:", sim_pos_neg_1)
print("Cosine Similarity between two negative reviews:", sim_neg_neg)


Cosine Similarity between two positive reviews: 0.751774204125345
Cosine Similarity between positive and negative review: 0.7570898021429142
Cosine Similarity between two negative reviews: 0.735549898323336


### Q1.C - discussion
Write your discussion here

### Q1.D - training and comparing classifiers (15 points)

For this task, you will use the `user_review_train.csv` and `user_review_test.csv` files to train a binary classification model with Word2Vec representations, and compare its performance with a binary classifier using Bag-of-Words representation.

As the Bag-of-Words classifier, you can either choose the best performing classifier you have implemented in Question 3 of Programming Assignment 1, or you can follow these steps:

- Preprocess the review with the given `preprocess_review` function.
- Order all unique tokens by frequency, take the most frequent 100.
- Use these 100 words as the corpus for Bag-of-Words representation.

For the Word2Vec model, represent the reviews by following these steps:

- Preprocess the review with the given `preprocess_review` function.
- For each token in the review that is also in the most frequent 100 tokens, fetch the vector of that token.
- Take the average of the token vectors selected to represent that review.

After training both classifiers on `user_review_train.csv`, test them with `user_review_test.csv` and report the performance of your models with four metrics: accuracy, precision, recall and F1-score. Compare the performance of both models and discuss in detail.

#### Notes and tips

- You can use `CountVectorizer` from `scikit-learn` or any other library available for Bag-of-Words representation.
- You should select a classification method from the following set of classifiers: `[Naive Bayes, Support Vector Machine, Logistic Regression, Random Forest]`. You can use `scikit-learn`, `nltk`, or any other library for the classifier implementations. 
- You should **not** use the test set `user_reviews_test.csv` during your training process. You should use `user_reviews_train.csv` only.
- You may add a validation step in your training process. To do this, you can further split the `user_reviews_train.csv` data and apply k-fold cross validation.

In [9]:
# Q1.D - implementation
# you can add cells below if needed

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from collections import Counter

# Load train and test data
train_data = pd.read_csv('data/user_review_train.csv')
test_data = pd.read_csv('data/user_review_test.csv')

# Preprocess all reviews
train_data["preprocessed_review"] = train_data["review"].apply(preprocess_review)
test_data["preprocessed_review"] = test_data["review"].apply(preprocess_review)

print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")

# Find most frequent 100 tokens from training data
all_tokens = [token for review in train_data['preprocessed_review'] for token in review]
token_freq = Counter(all_tokens)
top_100_tokens = [token for token, freq in token_freq.most_common(100)]

print(f"\nTop 100 most frequent tokens: {top_100_tokens[:10]}...")  # Show first 10

Training samples: 14675
Test samples: 1675

Top 100 most frequent tokens: ['.', 'phone', ',', 'good', 'camera', 'battery', 'mobile', '...', '..', 'product']...


In [10]:
# ===== BAG-OF-WORDS CLASSIFIER =====

# Create Bag-of-Words representation using top 100 tokens
def create_bow_vector(review, top_tokens):
    vector = np.zeros(len(top_tokens))
    for i, token in enumerate(top_tokens):
        vector[i] = review.count(token)
    return vector

# Create BOW representations
X_train_bow = np.array([create_bow_vector(review, top_100_tokens) for review in train_data['preprocessed_review']])
X_test_bow = np.array([create_bow_vector(review, top_100_tokens) for review in test_data['preprocessed_review']])
y_train = train_data['sentiment'].values
y_test = test_data['sentiment'].values

print(f"\nBOW Training set shape: {X_train_bow.shape}")
print(f"BOW Test set shape: {X_test_bow.shape}")

# Train Logistic Regression classifier with BOW
bow_classifier = LogisticRegression(max_iter=1000, random_state=42)
bow_classifier.fit(X_train_bow, y_train)

# Predict and evaluate BOW classifier
y_pred_bow = bow_classifier.predict(X_test_bow)

bow_accuracy = accuracy_score(y_test, y_pred_bow)
bow_precision = precision_score(y_test, y_pred_bow)
bow_recall = recall_score(y_test, y_pred_bow)
bow_f1 = f1_score(y_test, y_pred_bow)

print("\n===== BAG-OF-WORDS CLASSIFIER RESULTS =====")
print(f"Accuracy:  {bow_accuracy:.4f}")
print(f"Precision: {bow_precision:.4f}")
print(f"Recall:    {bow_recall:.4f}")
print(f"F1-Score:  {bow_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_bow, target_names=['Negative', 'Positive']))


BOW Training set shape: (14675, 100)
BOW Test set shape: (1675, 100)

===== BAG-OF-WORDS CLASSIFIER RESULTS =====
Accuracy:  0.8119
Precision: 0.8313
Recall:    0.8442
F1-Score:  0.8377

Classification Report:
              precision    recall  f1-score   support

    Negative       0.78      0.77      0.78       712
    Positive       0.83      0.84      0.84       963

    accuracy                           0.81      1675
   macro avg       0.81      0.81      0.81      1675
weighted avg       0.81      0.81      0.81      1675



In [11]:
# ===== WORD2VEC CLASSIFIER =====

# Create Word2Vec representation using top 100 tokens
def create_w2v_vector(review, model, top_tokens):
    """
    For each token in the review that is also in the top 100 tokens,
    fetch the vector and average them.
    """
    vectors = []
    for token in review:
        if token in top_tokens and token in model.wv:
            vectors.append(model.wv[token])
    
    if len(vectors) > 0:
        return np.mean(vectors, axis=0)
    else:
        # Return zero vector if no tokens found
        return np.zeros(model.wv.vector_size)

# Create Word2Vec representations
print("\nCreating Word2Vec representations...")
X_train_w2v = np.array([create_w2v_vector(review, model, top_100_tokens) for review in train_data['preprocessed_review']])
X_test_w2v = np.array([create_w2v_vector(review, model, top_100_tokens) for review in test_data['preprocessed_review']])

print(f"Word2Vec Training set shape: {X_train_w2v.shape}")
print(f"Word2Vec Test set shape: {X_test_w2v.shape}")

# Train Logistic Regression classifier with Word2Vec
w2v_classifier = LogisticRegression(max_iter=1000, random_state=42)
w2v_classifier.fit(X_train_w2v, y_train)

# Predict and evaluate Word2Vec classifier
y_pred_w2v = w2v_classifier.predict(X_test_w2v)

w2v_accuracy = accuracy_score(y_test, y_pred_w2v)
w2v_precision = precision_score(y_test, y_pred_w2v)
w2v_recall = recall_score(y_test, y_pred_w2v)
w2v_f1 = f1_score(y_test, y_pred_w2v)

print("\n===== WORD2VEC CLASSIFIER RESULTS =====")
print(f"Accuracy:  {w2v_accuracy:.4f}")
print(f"Precision: {w2v_precision:.4f}")
print(f"Recall:    {w2v_recall:.4f}")
print(f"F1-Score:  {w2v_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_w2v, target_names=['Negative', 'Positive']))


Creating Word2Vec representations...
Word2Vec Training set shape: (14675, 100)
Word2Vec Test set shape: (1675, 100)

===== WORD2VEC CLASSIFIER RESULTS =====
Accuracy:  0.8000
Precision: 0.8458
Recall:    0.7975
F1-Score:  0.8210

Classification Report:
              precision    recall  f1-score   support

    Negative       0.75      0.80      0.77       712
    Positive       0.85      0.80      0.82       963

    accuracy                           0.80      1675
   macro avg       0.80      0.80      0.80      1675
weighted avg       0.80      0.80      0.80      1675



In [12]:
# ===== COMPARISON =====

print("\n" + "="*60)
print("PERFORMANCE COMPARISON")
print("="*60)

comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Bag-of-Words': [bow_accuracy, bow_precision, bow_recall, bow_f1],
    'Word2Vec': [w2v_accuracy, w2v_precision, w2v_recall, w2v_f1]
})

print(comparison_df.to_string(index=False))

print("\n" + "="*60)
print("PERFORMANCE DIFFERENCES (Word2Vec - BOW)")
print("="*60)
print(f"Accuracy:  {w2v_accuracy - bow_accuracy:+.4f}")
print(f"Precision: {w2v_precision - bow_precision:+.4f}")
print(f"Recall:    {w2v_recall - bow_recall:+.4f}")
print(f"F1-Score:  {w2v_f1 - bow_f1:+.4f}")


PERFORMANCE COMPARISON
   Metric  Bag-of-Words  Word2Vec
 Accuracy      0.811940  0.800000
Precision      0.831288  0.845815
   Recall      0.844237  0.797508
 F1-Score      0.837713  0.820951

PERFORMANCE DIFFERENCES (Word2Vec - BOW)
Accuracy:  -0.0119
Precision: +0.0145
Recall:    -0.0467
F1-Score:  -0.0168


### Q1.D - discussion

**Performance Comparison:**

Both classifiers were trained using Logistic Regression on the same training and test sets. The key difference lies in the feature representation:

**Bag-of-Words (BOW) Classifier:**
- Uses frequency counts of the top 100 most frequent tokens
- Simple and interpretable representation
- Captures word presence/frequency but ignores semantic relationships
- Results: [Insert your results here after running]

**Word2Vec Classifier:**
- Uses averaged word embeddings from the top 100 most frequent tokens
- Captures semantic similarity between words
- Dense representation (100-dimensional vectors vs. sparse 100-dimensional counts)
- Results: [Insert your results here after running]

**Discussion Points:**

1. **Semantic Understanding**: Word2Vec should theoretically perform better because it captures semantic relationships (e.g., "excellent" and "great" have similar vectors), while BOW treats them as independent features.

2. **Information Loss**: Averaging Word2Vec vectors can lose important sentiment information, especially when positive and negative words appear together in a review.

3. **Feature Space**: BOW uses sparse binary/count features, while Word2Vec uses dense continuous features that may generalize better with limited data.

4. **Vocabulary Coverage**: Restricting to top 100 tokens might hurt Word2Vec more, as it benefits from a richer vocabulary to leverage semantic relationships.

5. **Training Data Size**: With the current dataset size, simpler BOW representation might perform comparably or even better than Word2Vec due to less complexity and overfitting.

**Conclusion:**
[Add your conclusion based on the actual results - which model performed better and why?]

## Q2 - Neural Networks for Binary Classification (50 points)

For this task, you will use the `user_review_train.csv` and `user_review_test.csv` files to train two neural network models for the binary classification of user reviews and compare their performances. You are expected to train RNN (part A - 20 points) and TextCNN (part B - 20 points) models, and report the following: 

- Confusion matrix of both models
- Time it took to train both models
- Accuracy, precision, recall, and F1-score of both models
- Other metrics you think are important

Finally (part C), you should discuss the performance of the models according to your reported results. Try to analyse the models in terms of pros and cons of using each one.

#### Notes and tips

- For the embedding layers of the models, you are free to use word embedding methods or leave them randomly initialised. Similarly, you can use word-based or character-based embeddings. However, make sure to explain your decisions.
- You are expected to use `tensorflow` for your implementations, but you can use other libraries if you already have a working setup.

In [ ]:
# Q2.A - implementation of RNN
# you can add cells below if needed



In [ ]:
# Q2.B - implementation of TextCNN
# you can add cells below if needed



### Q2.C - discussion

Write your discussion here.